# Feature Engineering & Preprocessing

Final preprocessing pipeline for both datasets: encoding, scaling, train/test split, and SMOTE resampling.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

fraud_df = pd.read_csv('../data/processed/fraud_data_clean.csv')
cc_df = pd.read_csv('../data/processed/creditcard_clean.csv')

print('Fraud_Data shape:', fraud_df.shape)
print('creditcard shape:', cc_df.shape)

## 1. E-Commerce Dataset (Fraud_Data)

In [ ]:
# Drop columns not useful for modeling
fraud_model = fraud_df.drop(columns=[
    'user_id', 'device_id', 'ip_address', 'signup_time', 'purchase_time'
])

# One-hot encode categoricals
categorical_cols = ['source', 'browser', 'sex', 'country']
fraud_model = pd.get_dummies(fraud_model, columns=categorical_cols, drop_first=True)

print('Shape after encoding:', fraud_model.shape)
fraud_model.head()

In [ ]:
# Separate features and target
X_fraud = fraud_model.drop(columns=['class'])
y_fraud = fraud_model['class']

# Stratified train/test split
X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    X_fraud, y_fraud, test_size=0.2, random_state=42, stratify=y_fraud
)

print('Train class distribution:')
print(y_train_f.value_counts())
print('\nTest class distribution:')
print(y_test_f.value_counts())

In [ ]:
# Scale numerical features
num_cols = ['purchase_value', 'age', 'time_since_signup', 'tx_count_24h']
num_cols_present = [c for c in num_cols if c in X_train_f.columns]

scaler_f = StandardScaler()
X_train_f[num_cols_present] = scaler_f.fit_transform(X_train_f[num_cols_present])
X_test_f[num_cols_present] = scaler_f.transform(X_test_f[num_cols_present])

print('Scaling applied to:', num_cols_present)

In [ ]:
# Apply SMOTE to training set only
print('Before SMOTE:', y_train_f.value_counts().to_dict())

smote = SMOTE(random_state=42)
X_train_f_res, y_train_f_res = smote.fit_resample(X_train_f, y_train_f)

print('After SMOTE:', pd.Series(y_train_f_res).value_counts().to_dict())

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
y_train_f.value_counts().plot(kind='bar', ax=axes[0], color=['steelblue', 'tomato'], title='Before SMOTE')
pd.Series(y_train_f_res).value_counts().plot(kind='bar', ax=axes[1], color=['steelblue', 'tomato'], title='After SMOTE')
for ax in axes:
    ax.set_xticklabels(['Legitimate', 'Fraud'], rotation=0)
plt.tight_layout()
plt.show()

## 2. Credit Card Dataset

In [ ]:
X_cc = cc_df.drop(columns=['Class'])
y_cc = cc_df['Class']

X_train_cc, X_test_cc, y_train_cc, y_test_cc = train_test_split(
    X_cc, y_cc, test_size=0.2, random_state=42, stratify=y_cc
)

print('Train class distribution:')
print(y_train_cc.value_counts())

In [ ]:
# Scale Amount and Time (V1-V28 are already PCA-scaled)
scaler_cc = StandardScaler()
X_train_cc[['Amount', 'Time']] = scaler_cc.fit_transform(X_train_cc[['Amount', 'Time']])
X_test_cc[['Amount', 'Time']] = scaler_cc.transform(X_test_cc[['Amount', 'Time']])

In [ ]:
# SMOTE for credit card — dataset is large so this may take a minute
# Justified: synthetic samples better represent fraud patterns than simple duplication
print('Before SMOTE:', y_train_cc.value_counts().to_dict())

smote_cc = SMOTE(random_state=42)
X_train_cc_res, y_train_cc_res = smote_cc.fit_resample(X_train_cc, y_train_cc)

print('After SMOTE:', pd.Series(y_train_cc_res).value_counts().to_dict())

## 3. Save Processed Splits

In [ ]:
# Save fraud data splits
X_train_f_res.to_csv('../data/processed/fraud_X_train.csv', index=False)
X_test_f.to_csv('../data/processed/fraud_X_test.csv', index=False)
pd.Series(y_train_f_res).to_csv('../data/processed/fraud_y_train.csv', index=False)
y_test_f.to_csv('../data/processed/fraud_y_test.csv', index=False)

# Save credit card splits
X_train_cc_res.to_csv('../data/processed/cc_X_train.csv', index=False)
X_test_cc.to_csv('../data/processed/cc_X_test.csv', index=False)
pd.Series(y_train_cc_res).to_csv('../data/processed/cc_y_train.csv', index=False)
y_test_cc.to_csv('../data/processed/cc_y_test.csv', index=False)

print('All splits saved to data/processed/')